# 🛡️ CodeSentinel AI — Train Model 3: Automated Refactoring Engine
### Fine-Tuning Salesforce CodeT5-Small on CodeXGLUE Code Refinement (Free T4 GPU in Google Colab)

This notebook trains **Model 3 (The Automated Code Refactoring & Repair Engine)** on the **CodeXGLUE Code Refinement** benchmark (~65,000+ buggy $\rightarrow$ clean code pairs).

* **Task**: Sequence-to-Sequence (Input: Smelly/Buggy Code $\rightarrow$ Output: Clean, Refactored Code)
* **Hardware**: Free Google Colab T4 GPU (16 GB VRAM)
* **Base Model**: `Salesforce/codet5-small` (~60M parameters, lightweight for free CPU deployment)
* **Output**: Pushed directly to your free Hugging Face account: `wraithklu/codesentinel-codet5-refiner`.

## 1. Verify GPU Hardware
Ensure you have selected **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
!pip install --quiet transformers datasets accelerate huggingface_hub

## 3. Load CodeXGLUE Code Refinement Dataset

In [ ]:
from datasets import load_dataset

print("📥 Loading CodeXGLUE Code Refinement dataset (subset: medium)...")
dataset = load_dataset("google/code_x_glue_cc_code_refinement", "medium")
print(dataset)
print("\nSample Buggy Code:")
print(dataset["train"][0]["buggy"])
print("\nSample Fixed/Clean Code:")
print(dataset["train"][0]["fixed"])

## 4. Setup CodeT5 Tokenizer & Seq2Seq Preprocessing

In [ ]:
import tokenizers
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

# Fix for Python 3.13: Rust tokenizers rejects transformers.AddedToken objects.
# Store the REAL Rust method on a class attribute (safe to re-run this cell).
if not hasattr(tokenizers.Tokenizer, '_real_add_tokens'):
    tokenizers.Tokenizer._real_add_tokens = tokenizers.Tokenizer.add_tokens
    tokenizers.Tokenizer._real_add_special = tokenizers.Tokenizer.add_special_tokens

def _fixed_add_tokens(self, tokens):
    if not isinstance(tokens, (list, tuple)):
        tokens = [tokens] if tokens is not None else []
    clean = [str(t.content) if hasattr(t, 'content') else str(t) for t in tokens]
    return tokenizers.Tokenizer._real_add_tokens(self, clean)

def _fixed_add_special(self, tokens):
    if not isinstance(tokens, (list, tuple)):
        tokens = [tokens] if tokens is not None else []
    clean = [str(t.content) if hasattr(t, 'content') else str(t) for t in tokens]
    return tokenizers.Tokenizer._real_add_special(self, clean)

tokenizers.Tokenizer.add_tokens = _fixed_add_tokens
tokenizers.Tokenizer.add_special_tokens = _fixed_add_special

MODEL_ID = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    inputs = ["refactor code: " + code for code in examples["buggy"]]
    targets = examples["fixed"]

    model_inputs = tokenizer(inputs, max_length=256, truncation=True)
    labels = tokenizer(text_target=targets, max_length=256, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("🔄 Tokenizing dataset for Seq2Seq training...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("✅ Tokenization complete!")

## 5. Load CodeT5 Model & Configure Trainer

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./refactor_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  # 16-bit mixed precision on T4 GPU
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

## 6. Train the Refactoring Model (Estimated: ~35-40 mins on T4 GPU)

In [ ]:
print("🚀 Starting Training on CodeT5-Small...")
trainer.train()

## 7. Test Refactoring on a Sample Code Snippet

In [ ]:
sample_buggy_code = "public int add(int a, int b) { int c = a + b; return c; }"
input_text = "refactor code: " + sample_buggy_code

input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
generated_ids = model.generate(input_ids, max_length=128, num_beams=3, early_stopping=True)
refactored_code = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Original Code:")
print(sample_buggy_code)
print("\nRefactored Code by CodeT5:")
print(refactored_code)

## 8. Export & Push to Hugging Face Model Hub

In [ ]:
from huggingface_hub import notebook_login

print("🔑 Sign in to Hugging Face:")
notebook_login()

In [ ]:
HF_USERNAME = "wraithklu"  # Replace with your Hugging Face username
REPO_NAME = f"{HF_USERNAME}/codesentinel-codet5-refiner"

print(f"📤 Pushing refactor model to: https://huggingface.co/{REPO_NAME}")
trainer.save_model("./codet5-refiner")
tokenizer.save_pretrained("./codet5-refiner")

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"🎉 SUCCESS! Model is live at https://huggingface.co/{REPO_NAME}")